# Mastermind: Unicode-Zeichen und ipywidgets

In diesem Notebook lernst du zwei Werkzeuge kennen, mit denen du dein Mastermind-Spiel darstellen und bedienen kannst:

1. **Unicode-Zeichen**: farbige Kreise (🔴🔵🟢) und Feedback-Stifte (●○) als ganz normaler Text
2. **ipywidgets**: Buttons und Auswahlmenüs, damit du das Spiel per Klick statt per Tastatur spielst

**Ablauf:** Teil 1 (Unicode) → Teil 2 (ipywidgets) → Teil 3 (beides im Mastermind kombinieren). Führe jede Code-Zelle mit `Shift + Enter` aus und probiere die Aufgaben selbst.

## Teil 1: Unicode-Zeichen

Ein Unicode-Zeichen wie 🔴 oder ● ist für Python ein **String**, genau wie ein Buchstabe. Du kannst es in Variablen, Listen und Dictionaries speichern, mit `print` ausgeben und mit `*` wiederholen.

### 1.1 Wo hole ich die Zeichen?

**a) Emoji-Tastatur des Betriebssystems**

| System | Tastenkombination |
|---|---|
| Windows | `Windows` + `.` (Punkt) |
| macOS | `Ctrl` + `Cmd` + `Leertaste` |
| iPad / Handy | Emoji-Tastatur, Suchbegriff «circle» oder «Kreis» |

**b) Zeichentabellen im Web** (Zeichen anklicken, kopieren, in den Code einfügen)

- <https://emojipedia.org>: alle Emoji, z.B. Suche nach «circle»
- <https://symbl.cc> oder <https://unicode-table.com>: Symbole wie ● ○ • ·

**c) Über den Code-Punkt.** Jedes Zeichen hat eine Nummer, z.B. `U+1F534` für 🔴. Damit funktioniert es auf jedem Gerät, auch ohne Emoji-Tastatur.

**d) Über den Namen.** Jedes Zeichen hat einen offiziellen englischen Namen.

In [ ]:
print("🔴")                      # direkt eingefügt (Copy & Paste)
print(chr(0x1F534))             # über den Code-Punkt
print("\U0001F534")             # Escape-Schreibweise
print("\N{LARGE RED CIRCLE}")   # über den Namen

### 1.2 Code-Punkt und Namen eines Zeichens herausfinden

Kopiere ein Zeichen in die Variable `zeichen`. Python verrät dir dann seine Nummer und seinen Namen.

In [ ]:
import unicodedata

zeichen = "🟢"
print(hex(ord(zeichen)))            # Code-Punkt: 0x1f7e2
print(unicodedata.name(zeichen))    # Name: LARGE GREEN CIRCLE

### 1.3 Die wichtigsten Zeichen für Mastermind

| Zeichen | Code-Punkt | Verwendung |
|---|---|---|
| 🔴 | `U+1F534` | Farbe rot |
| 🔵 | `U+1F535` | Farbe blau |
| 🟢 | `U+1F7E2` | Farbe grün |
| 🟡 | `U+1F7E1` | Farbe gelb |
| 🟠 | `U+1F7E0` | Farbe orange |
| 🟣 | `U+1F7E3` | Farbe lila |
| 🟤 | `U+1F7E4` | Farbe braun |
| ⚫ | `U+26AB` | Feedback: richtige Farbe, richtiger Platz (gross) |
| ⚪ | `U+26AA` | Feedback: richtige Farbe, falscher Platz (gross) |
| ● | `U+25CF` | Feedback schwarz (klein, Textzeichen) |
| ○ | `U+25CB` | Feedback weiss (klein, Textzeichen) |
| · | `U+00B7` | Platzhalter «kein Treffer» |

**Hinweis:** Emoji (🔴) sind grösser als Textzeichen (●). Für die Feedback-Stifte sind die kleinen Textzeichen praktischer. Wie ein Zeichen genau aussieht, hängt von Gerät und Browser ab.

### 1.4 Unicode-Zeichen in Python verwenden

Ein **Dictionary** verbindet den Farbnamen mit dem Zeichen. So rechnest du im Programm mit Namen und zeigst trotzdem Kreise an.

In [ ]:
FARBEN = {
    "rot": "🔴",
    "blau": "🔵",
    "grün": "🟢",
    "gelb": "🟡",
    "orange": "🟠",
    "lila": "🟣",
}

versuch = ["rot", "blau", "grün", "gelb"]

# Liste von Farbnamen -> Zeile mit Kreisen
print(" ".join(FARBEN[f] for f in versuch))

Das **Feedback** besteht aus schwarzen Stiften (richtige Farbe *und* Position), weissen Stiften (richtige Farbe, falsche Position) und Platzhaltern. Mit `*` kannst du ein Zeichen wiederholen:

In [ ]:
def feedback_text(schwarz, weiss, laenge=4):
    leer = laenge - schwarz - weiss
    return "●" * schwarz + "○" * weiss + "·" * leer

print(feedback_text(2, 1))   # ●●○·
print(feedback_text(0, 0))   # ····
print(feedback_text(4, 0))   # ●●●●

Die **Bewertung** eines Versuchs: Zuerst zählen wir die Treffer am richtigen Platz. Dann zählen wir für jede Farbe, wie oft sie in beiden Codes vorkommt (das Minimum), und ziehen die schwarzen Treffer wieder ab.

In [ ]:
def bewerte(geheim, versuch):
    schwarz = sum(g == v for g, v in zip(geheim, versuch))
    gemeinsam = sum(min(geheim.count(f), versuch.count(f)) for f in set(geheim))
    weiss = gemeinsam - schwarz
    return schwarz, weiss

geheim = ["rot", "blau", "grün", "gelb"]
versuch = ["rot", "grün", "rot", "orange"]

s, w = bewerte(geheim, versuch)
print(" ".join(FARBEN[f] for f in versuch), "  ", feedback_text(s, w))

### Aufgaben zu Teil 1

1. Suche mit einer der Methoden a) bis d) ein Zeichen für **braun** 🟤 und füge es zum Dictionary `FARBEN` hinzu.
2. Finde mit `unicodedata.name()` heraus, wie das Zeichen `●` offiziell heisst.
3. Erzeuge einen zufälligen Geheimcode aus 4 Farben. Tipp: `random.choice(list(FARBEN))`, in einer Schleife oder List Comprehension.
4. Gib den Geheimcode mit Kreisen aus und teste `bewerte` mit selbst gewählten Versuchen. Stimmt das Feedback immer? Probiere auch Codes mit doppelten Farben, z.B. `["rot", "rot", "blau", "blau"]`.

## Teil 2: ipywidgets

Bis jetzt hast du deine Versuche im Code getippt. Mit **ipywidgets** baust du bedienbare Elemente ins Notebook: Buttons, Auswahlmenüs, Schieberegler.

Das Prinzip heisst **ereignisgesteuert**: Du legst fest, *was passieren soll*, wenn jemand klickt oder etwas auswählt. Python führt die Funktion dann aus, sobald das Ereignis eintritt.

### 2.1 Import

```python
import ipywidgets as widgets
from IPython.display import display
```

Falls eine Fehlermeldung `ModuleNotFoundError` erscheint, ist ipywidgets nicht installiert. Melde dich dann bei deiner Lehrperson.

### 2.2 Der Button

Ein Button braucht zwei Dinge: das Widget selbst und eine **Funktion**, die beim Klick aufgerufen wird. Die Funktion bekommt automatisch den Button als Argument (hier `b`), auch wenn wir ihn nicht brauchen.

Ausgaben mit `print` schreiben wir in ein `Output`-Widget. Sonst erscheinen sie an einer unpassenden Stelle.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

knopf = widgets.Button(description="Klick mich!", button_style="success")
ausgabe = widgets.Output()
zaehler = 0

def beim_klick(b):
    global zaehler
    zaehler += 1
    with ausgabe:
        print(f"Geklickt: {zaehler}x")

knopf.on_click(beim_klick)     # Funktion mit dem Button verbinden

display(knopf, ausgabe)

`button_style` kennt `"success"` (grün), `"info"` (blau), `"warning"` (orange), `"danger"` (rot) und `""` (grau).

### 2.3 Das Dropdown-Menü

Bei einem `Dropdown` legst du die Auswahl mit `options` fest. Den gewählten Wert liest du mit `.value`.

In [ ]:
farbwahl = widgets.Dropdown(options=["rot", "blau", "grün"], description="Farbe:")
display(farbwahl)

In [ ]:
# Wähle oben eine Farbe aus und führe dann diese Zelle aus:
print(farbwahl.value)

Mit **Paaren** `(Anzeige, Wert)` zeigt das Menü etwas anderes an, als du später im Programm brauchst. Das ist ideal für Mastermind: Der Kreis wird angezeigt, im Programm rechnest du mit dem Farbnamen.

In [ ]:
farbwahl2 = widgets.Dropdown(
    options=[("🔴 rot", "rot"), ("🔵 blau", "blau"), ("🟢 grün", "grün")]
)
display(farbwahl2)

In [ ]:
print(farbwahl2.value)   # gibt "rot", "blau" oder "grün" zurück, nicht das Emoji

### 2.4 Widgets anordnen

`HBox` ordnet Widgets **nebeneinander** an (horizontal), `VBox` **untereinander** (vertikal). Sie lassen sich verschachteln.

In [ ]:
d1 = widgets.Dropdown(options=["rot", "blau", "grün"])
d2 = widgets.Dropdown(options=["rot", "blau", "grün"])
d3 = widgets.Dropdown(options=["rot", "blau", "grün"])

display(widgets.HBox([d1, d2, d3]))

### 2.5 Weitere nützliche Widgets

| Widget | Wofür? | Wert lesen |
|---|---|---|
| `widgets.Button` | Aktion auslösen | `on_click(funktion)` |
| `widgets.Dropdown` | eine Option aus einer Liste | `.value` |
| `widgets.ToggleButtons` | eine Option per Knopfreihe | `.value` |
| `widgets.RadioButtons` | eine Option mit Auswahlpunkten | `.value` |
| `widgets.IntSlider` | Zahl per Schieberegler | `.value` |
| `widgets.Text` | Text eintippen | `.value` |
| `widgets.Output` | Ausgaben (`print`) darstellen | `with ausgabe: ...` |
| `widgets.HBox` / `VBox` | Anordnung | |

### Aufgaben zu Teil 2

1. Erstelle einen Button «Würfeln», der bei jedem Klick eine Zufallszahl von 1 bis 6 ausgibt.
2. Erstelle ein Dropdown mit den Farben 🔴 🔵 🟢 und einen Button «Anzeigen», der die gewählte Farbe ausgibt.
3. Ersetze das Dropdown durch `widgets.ToggleButtons`. Was ändert sich?
4. Stelle mit `IntSlider(min=1, max=10, value=5)` eine Zahl ein und gib sie beim Klick auf einen Button aus.

## Teil 3: Mastermind mit ipywidgets

Jetzt kombinierst du alles. Das Spiel besteht aus fünf Bausteinen:

| Baustein | Umsetzung |
|---|---|
| Geheimcode | Liste mit 4 zufälligen Farbnamen |
| Eingabe | 4 Dropdowns mit Farb-Emoji, in einer `HBox` |
| Auslöser | Button «Raten» |
| Auswertung | Funktion `bewerte` aus Teil 1 |
| Anzeige | `Output`-Widget mit Kreisen und Feedback-Stiften |

### 3.1 Vorbereitung

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display

FARBEN = {"rot": "🔴", "blau": "🔵", "grün": "🟢",
          "gelb": "🟡", "orange": "🟠", "lila": "🟣"}
LAENGE = 4
MAX_VERSUCHE = 10

def feedback_text(schwarz, weiss, laenge=LAENGE):
    leer = laenge - schwarz - weiss
    return "●" * schwarz + "○" * weiss + "·" * leer

def bewerte(geheim, versuch):
    schwarz = sum(g == v for g, v in zip(geheim, versuch))
    gemeinsam = sum(min(geheim.count(f), versuch.count(f)) for f in set(geheim))
    return schwarz, gemeinsam - schwarz

# Geheimcode und Versuchszähler
geheim = [random.choice(list(FARBEN)) for _ in range(LAENGE)]
versuche = 0

### 3.2 Eingabe: vier Dropdowns

Wir erzeugen die Dropdowns mit einer List Comprehension. Jedes zeigt Kreis und Name an und liefert den Farbnamen als Wert.

In [ ]:
optionen = [(f"{symbol} {name}", name) for name, symbol in FARBEN.items()]

auswahl = [
    widgets.Dropdown(options=optionen, layout=widgets.Layout(width="110px"))
    for _ in range(LAENGE)
]

raten_knopf = widgets.Button(description="Raten", button_style="success")
ausgabe = widgets.Output()

### 3.3 Die Funktion beim Klick

Bei jedem Klick passiert Folgendes:

1. Farbnamen aus den vier Dropdowns lesen
2. Versuch bewerten
3. Zeile mit Kreisen und Feedback ausgeben
4. Prüfen, ob gewonnen oder verloren

In [ ]:
def beim_raten(b):
    global versuche
    versuch = [d.value for d in auswahl]          # 1. lesen
    versuche += 1
    schwarz, weiss = bewerte(geheim, versuch)     # 2. bewerten

    kreise = " ".join(FARBEN[f] for f in versuch)
    with ausgabe:                                 # 3. anzeigen
        print(f"{versuche:2}.  {kreise}    {feedback_text(schwarz, weiss)}")

        if schwarz == LAENGE:                     # 4. Spielende prüfen
            print(f"🎉 Geschafft in {versuche} Versuchen!")
            raten_knopf.disabled = True
        elif versuche >= MAX_VERSUCHE:
            print("Leider verloren. Der Code war:", " ".join(FARBEN[f] for f in geheim))
            raten_knopf.disabled = True

raten_knopf.on_click(beim_raten)

display(widgets.VBox([widgets.HBox(auswahl), raten_knopf, ausgabe]))

**Wichtig:** Führe Zelle 3.1 nur einmal aus. Sonst wird ein neuer Geheimcode erzeugt und der Zähler zurückgesetzt. Führst du die Zelle mit `display(...)` mehrmals aus, hängst du den Button doppelt an die Funktion: Ein Klick zählt dann mehrfach. In diesem Fall beim Ausführen immer von Zelle 3.1 an neu starten.

### 3.4 Erweiterungen

**Grundaufgaben**

1. Zeige oberhalb der Eingabe an, wie viele Versuche noch übrig sind. Tipp: `widgets.Label` und `.value` ändern.
2. Füge einen Button **«Neues Spiel»** hinzu. Er erzeugt einen neuen Geheimcode, setzt `versuche` auf 0, leert die Ausgabe (`ausgabe.clear_output()`) und aktiviert den Raten-Button wieder.
3. Ändere `LAENGE` auf 5 oder verwende nur 4 Farben. Was musst du im Code anpassen?

**Fortgeschritten**

4. Ersetze die Dropdowns durch `widgets.ToggleButtons`. Vergleiche die Bedienung.
5. Mit einem `IntSlider` stellt der Spieler die Anzahl erlaubter Versuche ein.
6. Ersetze die Unicode-Feedbackstifte durch graue und schwarze Kreise mit `widgets.HTML`:
   `widgets.HTML('<span style="color:gray">●</span>')`
7. Zwei-Spieler-Modus: Spieler A wählt den Geheimcode mit Dropdowns, Spieler B rät.